In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Mga AI-powered transpiler pass

Ang mga AI-powered transpiler pass ay mga pass na gumagana bilang kapalit ng mga "tradisyonal" na Qiskit pass para sa ilang gawain sa transpiling. Kadalasan, mas maganda ang kanilang mga resulta kumpara sa mga umiiral na heuristic na algorithm (tulad ng mas mababang depth at CNOT count), at mas mabilis pa rin sila kaysa sa mga optimization algorithm tulad ng Boolean satisfiability solver. Ang mga AI transpiler pass ay tumatakbo sa iyong lokal na environment.

> **Note:** Ang mga AI-powered transpiler pass ay nasa beta release status pa, at maaaring magbago.
> Kung mayroon kang feedback o nais makipag-ugnayan sa development team, gamitin ang [Qiskit Slack Workspace channel](https://qiskit.slack.com/archives/C06KF8YHUAU) na ito.

Ang mga sumusunod na pass ay kasalukuyang available:

**Mga Routing pass**
 - `AIRouting`: Pagpili ng layout at pag-route ng circuit

**Mga circuit synthesis pass**
 - `AICliffordSynthesis`: Clifford circuit synthesis
 - `AILinearFunctionSynthesis`: Linear function circuit synthesis
 - `AIPermutationSynthesis`: Permutation circuit synthesis

Para gamitin ang mga AI transpiler pass, i-install muna ang `qiskit-ibm-transpiler` package. Bisitahin ang [qiskit-ibm-transpiler API documentation](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) para sa mas detalyadong impormasyon tungkol sa iba't ibang available na opsyon.

## Patakbuhin ang mga AI transpiler pass nang lokal o sa cloud
I-install muna ang `qiskit-ibm-transpiler` na may dagdag na dependencies tulad ng sumusunod:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService
import logging

backend = QiskitRuntimeService().backend("ibm_fez")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)
transpiled_circuit = ai_passmanager.run(circuit)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Pagkatapos i-install ang mga karagdagang dependencies, ang default na paraan para patakbuhin ang mga AI-powered transpiler pass ay ang gamitin ang iyong lokal na makina.

## AI routing pass
Ang `AIRouting` pass ay gumaganap bilang layout stage at routing stage. Maaari itong gamitin sa loob ng isang `PassManager` tulad ng sumusunod:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit.circuit.library import efficient_su2

ibm_kingston = QiskitRuntimeService().backend("ibm_kingston")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_kingston,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_kingston, local_mode=True
        ),  # Re-synthesize Linear Function blocks
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Fetching 127 files:   0%|          | 0/127 [00:00<?, ?it/s]

The synthesis respects the coupling map of the device: it can be run safely after other routing passes without disturbing the circuit, so the overall circuit will still follow the device restrictions. By default, the synthesis will replace the original sub-circuit only if the synthesized sub-circuit improves the original (currently only checking CNOT count), but this can be forced to always replace the circuit by setting `replace_only_if_better=False`.

The following synthesis passes are available from `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Synthesis for [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) circuits (blocks of `H`, `S`, and `CX` gates). Currently up to nine qubit blocks.
- *AILinearFunctionSynthesis*: Synthesis for [Linear Function](/docs/api/qiskit/qiskit.circuit.library.LinearFunction) circuits (blocks of `CX` and `SWAP` gates). Currently up to nine qubit blocks.
- *AIPermutationSynthesis*: Synthesis for [Permutation](/docs/api/qiskit/qiskit.circuit.library.Permutation#permutation) circuits (blocks of `SWAP` gates). Currently available for 65, 33, and 27 qubit blocks.
- *AIPauliNetworkSynthesis*: Synthesis for Pauli Network circuits (blocks of `H`, `S`, `SX`, `CX`, `RX`, `RY` and `RZ` gates). Currently up to six qubit blocks.

We expect to gradually increase the size of the supported blocks.

All passes use a thread pool to send several requests in parallel. By default, the number for max threads is the number of cores plus four (default values for the `ThreadPoolExecutor` Python object). However, you can set your own value with the `max_threads` argument at pass instantiation. For example, the following line instantiates the `AILinearFunctionSynthesis` pass, which allows it to use a maximum of 20 threads.

```python
AILinearFunctionSynthesis(backend=ibm_torino, max_threads=20)  # Re-synthesize Linear Function blocks using 20 threads max
```

You can also set the environment variable `AI_TRANSPILER_MAX_THREADS` to the desired number of maximum threads, and all synthesis passes instantiated after that will use that value.

For the AI synthesis passes to synthesize a sub-circuit, it must lay on a connected subgraph of the coupling map (one way to do this is with a routing pass before collecting the blocks, but this is not the only way to do it). The synthesis passes will automatically check that the specific subgraph is supported, and if not, it will raise a warning and leave the original sub-circuit unchanged.

The following custom collection passes for Cliffords, Linear Functions and Permutations that can be imported from `qiskit_ibm_transpiler.ai.collection` also complement the synthesis passes:

- *CollectCliffords*: Collects Clifford blocks as `Instruction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectLinearFunctions*: Collects blocks of `SWAP` and `CX` as `LinearFunction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectPermutations*: Collects blocks of `SWAP` circuits as `Permutations`.
- *CollectPauliNetworks*: Collects Pauli Network blocks and stores the original sub-circuit to compare against it after synthesis.

These custom collection passes limit the sizes of the collected sub-circuits so they are supported by the AI-powered synthesis passes. Therefore, it is recommended to use them after the routing passes and before the synthesis passes for a better overall optimization.

## Hybrid heuristic-AI circuit transpilation

The `qiskit-ibm-transpiler` allows you to configure a hybrid pass manager that combines the best of Qiskit's heuristic and the AI-powered transpiler passes. This feature behaves similarly to the Qiskit `generate_pass_manager` method. A typical way to use `generate_ai_pass_manager` is as follows:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_kingston")
kingston_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=kingston_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

Dito, tinutukoy ng `backend` kung aling coupling map ang gagamitin para sa routing, tinutukoy ng `optimization_level` (1, 2, o 3) kung gaano karaming computational effort ang gagamitin sa proseso (mas mataas ay karaniwang nagbibigay ng mas magandang resulta ngunit mas matagal), at tinutukoy ng `layout_mode` kung paano hawakan ang pagpili ng layout.
Kasama sa `layout_mode` ang mga sumusunod na opsyon:

- `keep`: Iginagalang nito ang layout na itinakda ng mga nakaraang transpiler pass (o gumagamit ng trivial layout kung hindi pa naitakda). Karaniwang ginagamit lamang ito kapag kailangang patakbuhin ang circuit sa mga partikular na qubit ng device. Kadalasan, nagbibigay ito ng mas masamang resulta dahil mas limitado ang pagkakataon para sa optimization.
- `improve`: Ginagamit nito ang layout na itinakda ng mga nakaraang transpiler pass bilang panimula. Kapaki-pakinabang ito kapag mayroon kang magandang paunang hulà para sa layout; halimbawa, para sa mga circuit na ginawa sa paraang halos sumusunod sa coupling map ng device. Kapaki-pakinabang din ito kung nais mong subukan ang iba pang partikular na layout pass kasama ang `AIRouting` pass.
- `optimize`: Ito ang default na mode. Pinakaepektibo ito para sa mga pangkalahatang circuit kung saan maaaring wala kang magandang hulà sa layout. Binabalewala ng mode na ito ang mga nakaraang pagpili ng layout.
- `local_mode`: Tinutukoy ng flag na ito kung saan tatakbo ang `AIRouting` pass. Kung `False`, ang `AIRouting` ay tatakbo nang remote sa pamamagitan ng Qiskit Transpiler Service. Kung `True`, susubukan ng package na patakbuhin ang pass sa iyong lokal na environment na may fallback sa cloud mode kung hindi mahanap ang mga kinakailangang dependencies.

## Mga AI circuit synthesis pass
Ang mga AI circuit synthesis pass ay nagbibigay-daan sa iyo na i-optimize ang mga bahagi ng iba't ibang uri ng circuit ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauli Network) sa pamamagitan ng muling pag-synthesize sa kanila. Ang isang tipikal na paraan para gamitin ang synthesis pass ay tulad ng sumusunod: